# UAS Bengkel Koding Data Science
## Prediksi Customer Churn — Sales and Marketing Dataset
**Universitas Dian Nuswantoro**

---
## 0. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import joblib

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
print('Libraries loaded!')

---
## 1. EDA (Exploratory Data Analysis)
### 1.1 Load Dataset

In [ ]:
# Jika di Google Colab: upload file lalu pakai path /content/
# Jika di VS Code lokal: taruh CSV di folder yang sama
df = pd.read_csv('Sales_-_Marketing_customer_dataset.csv')
print(f'Dataset shape: {df.shape}')
print(f'Total records: {len(df)}')

### 1.2 Tampilkan 5 Baris Pertama, Info, dan Statistik Deskriptif

In [ ]:
print('=== 5 BARIS PERTAMA ===')
df.head()

In [ ]:
print('=== INFORMASI DATASET ===')
df.info()

In [ ]:
print('=== STATISTIK DESKRIPTIF ===')
df.describe(include='all')

### 1.3 Missing Value Analysis

In [ ]:
# Hitung persentase missing value
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_all = missing_pct.sort_values(ascending=False)
missing_only = missing_all[missing_all > 0]

print('=== MISSING VALUE PER KOLOM (%) ===')
print(missing_only)
print(f'\nTotal kolom dengan missing value: {len(missing_only)}')

# Visualisasi semua kolom
fig, ax = plt.subplots(figsize=(16, 5))
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in missing_all.values]
bars = ax.bar(missing_all.index, missing_all.values, color=colors, edgecolor='black', width=0.7)
ax.set_title('Persentase Missing Value per Kolom', fontsize=14, fontweight='bold')
ax.set_xlabel('Kolom')
ax.set_ylabel('Persentase (%)')
ax.axhline(y=5, color='orange', linestyle='--', linewidth=1.5, label='Threshold 5%')
ax.legend()
plt.xticks(rotation=45, ha='right', fontsize=9)
for bar, val in zip(bars, missing_all.values):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}%', ha='center', fontsize=8, color='red', fontweight='bold')
plt.tight_layout()
plt.show()

### 1.4 Distribusi Variabel Target (Churn)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

churn_counts = df['churn'].value_counts().sort_index()
labels = ['Tidak Churn (0)', 'Churn (1)']
colors = ['#2ecc71', '#e74c3c']

# Bar chart
axes[0].bar(labels, churn_counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Distribusi Variabel Target Churn', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Jumlah Pelanggan')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 80, str(v), ha='center', fontweight='bold', fontsize=11)

# Pie chart
axes[1].pie(churn_counts.values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90, explode=(0, 0.05))
axes[1].set_title('Proporsi Churn vs Tidak Churn', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Tidak Churn (0): {churn_counts[0]:,} ({churn_counts[0]/len(df)*100:.1f}%)')
print(f'Churn       (1): {churn_counts[1]:,} ({churn_counts[1]/len(df)*100:.1f}%)')
print('\nDataset IMBALANCED — kelas 0 jauh lebih dominan dari kelas 1.')

### 1.5 Heatmap Korelasi Fitur Numerik

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'customer_id']
print(f'Kolom numerik ({len(numeric_cols)}): {numeric_cols}')

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, ax=ax, linewidths=0.5,
            annot_kws={'size': 8})
ax.set_title('Heatmap Korelasi Fitur Numerik', fontsize=15, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\n=== KORELASI FITUR TERHADAP CHURN (descending) ===')
churn_corr = corr_matrix['churn'].drop('churn').sort_values(key=abs, ascending=False)
print(churn_corr.round(4))

---
## 2. Direct Modeling (Tanpa Preprocessing & Tanpa Tuning)

### 2.1 Persiapan Data Direct

In [ ]:
# Direct modeling: pakai kolom numerik saja (hindari error pada kolom object)
# Isi missing value dengan median supaya bisa langsung di-train
df_direct = df.copy()

num_cols_direct = df_direct.select_dtypes(include=[np.number]).columns.tolist()
num_cols_direct = [c for c in num_cols_direct if c not in ['customer_id', 'churn']]

X_direct = df_direct[num_cols_direct].fillna(df_direct[num_cols_direct].median())
y_direct = df_direct['churn']

print(f'Fitur ({len(num_cols_direct)}): {num_cols_direct}')
print(f'Shape X: {X_direct.shape} | Shape y: {y_direct.shape}')

In [ ]:
# Train-test split 80:20 stratified
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_direct, y_direct, test_size=0.2, random_state=42, stratify=y_direct
)
print(f'Train: {X_train_d.shape} | Test: {X_test_d.shape}')
print(f'Distribusi train — 0: {(y_train_d==0).sum()}, 1: {(y_train_d==1).sum()}')

### 2.2 Helper Function Evaluasi

In [ ]:
def evaluate_model(model, X_test, y_test, model_name='Model'):
    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    cm   = confusion_matrix(y_test, y_pred)

    print(f'\n{"="*55}')
    print(f'  {model_name}')
    print(f'{"="*55}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print(f'\n{classification_report(y_test, y_pred, target_names=["Tidak Churn","Churn"])}')

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Tidak Churn','Churn'],
                yticklabels=['Tidak Churn','Churn'])
    ax.set_title(f'Confusion Matrix\n{model_name}', fontweight='bold')
    ax.set_ylabel('Aktual'); ax.set_xlabel('Prediksi')
    plt.tight_layout(); plt.show()

    return {'model': model_name, 'accuracy': acc,
            'precision': prec, 'recall': rec, 'f1': f1}

print('Helper function siap!')

### 2.3 Model 1 — Logistic Regression (Konvensional)

In [ ]:
lr_d = LogisticRegression(random_state=42, max_iter=1000)
lr_d.fit(X_train_d, y_train_d)
res_lr_d = evaluate_model(lr_d, X_test_d, y_test_d, 'Logistic Regression (Direct)')

### 2.4 Model 2 — Random Forest (Ensemble Bagging)

In [ ]:
rf_d = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_d.fit(X_train_d, y_train_d)
res_rf_d = evaluate_model(rf_d, X_test_d, y_test_d, 'Random Forest (Direct)')

### 2.5 Model 3 — Voting Classifier LR+SVM+KNN (Ensemble Voting)

In [ ]:
voting_d = VotingClassifier(estimators=[
    ('lr',  LogisticRegression(random_state=42, max_iter=1000)),
    ('svm', SVC(probability=True, random_state=42)),
    ('knn', KNeighborsClassifier())
], voting='soft')
voting_d.fit(X_train_d, y_train_d)
res_voting_d = evaluate_model(voting_d, X_test_d, y_test_d, 'Voting Classifier (Direct)')

### 2.6 Rangkuman Direct Modeling

In [ ]:
results_d = pd.DataFrame([res_lr_d, res_rf_d, res_voting_d]).set_index('model')
print('=== RANGKUMAN DIRECT MODELING ===')
print(results_d.round(4))

results_d.plot(kind='bar', figsize=(11, 5), colormap='Set2', edgecolor='black')
plt.title('Perbandingan Metrik — Direct Modeling', fontsize=13, fontweight='bold')
plt.ylabel('Score'); plt.ylim(0, 1.1)
plt.xticks(rotation=15, ha='right'); plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

---
## 3. Modeling dengan Preprocessing
### 3.1 Preprocessing Data

In [ ]:
df_prep = df.copy()

# 1. Drop kolom tidak relevan
drop_cols = ['customer_id', 'signup_date', 'last_purchase_date', 'coupon_code']
df_prep.drop(columns=drop_cols, inplace=True)
print(f'Setelah drop: {df_prep.shape}')

# 2. Hapus duplikat
dup = df_prep.duplicated().sum()
df_prep.drop_duplicates(inplace=True)
print(f'Duplikat dihapus: {dup}')

# 3. Handle missing value
# gender & satisfaction_score: isi dengan modus/median
for col in df_prep.columns:
    if df_prep[col].isnull().sum() > 0:
        if df_prep[col].dtype == 'object':
            fill_val = df_prep[col].mode()[0]
        else:
            fill_val = df_prep[col].median()
        df_prep[col].fillna(fill_val, inplace=True)
        print(f'  Imputasi [{col}] dengan: {fill_val}')

print(f'\nMissing setelah imputasi: {df_prep.isnull().sum().sum()}')

In [ ]:
# 4. Outlier handling — IQR capping pada fitur numerik
num_prep = df_prep.select_dtypes(include=[np.number]).columns
num_prep = [c for c in num_prep if c != 'churn']

total_outliers = 0
for col in num_prep:
    Q1, Q3 = df_prep[col].quantile(0.25), df_prep[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((df_prep[col] < lower) | (df_prep[col] > upper)).sum()
    total_outliers += n_out
    df_prep[col] = df_prep[col].clip(lower, upper)

print(f'Total outlier di-cap: {total_outliers}')

# 5. Encoding kolom kategorikal
cat_cols = df_prep.select_dtypes(include='object').columns.tolist()
print(f'Kolom kategorikal: {cat_cols}')
le = LabelEncoder()
for col in cat_cols:
    df_prep[col] = le.fit_transform(df_prep[col].astype(str))
print('Encoding selesai!')

In [ ]:
# Pisah X dan y
X_prep = df_prep.drop(columns=['churn'])
y_prep = df_prep['churn']

# Train-test split 80:20 (sama dengan direct)
X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(
    X_prep, y_prep, test_size=0.2, random_state=42, stratify=y_prep
)

# Scaling SETELAH split
scaler = StandardScaler()
X_tr_p_sc = scaler.fit_transform(X_tr_p)
X_te_p_sc = scaler.transform(X_te_p)

print(f'Train: {X_tr_p_sc.shape} | Test: {X_te_p_sc.shape}')
print(f'Fitur yang digunakan ({X_prep.shape[1]}): {X_prep.columns.tolist()}')

### 3.2 Model 1 — Logistic Regression (Preprocessing)

In [ ]:
lr_p = LogisticRegression(random_state=42, max_iter=1000)
lr_p.fit(X_tr_p_sc, y_tr_p)
res_lr_p = evaluate_model(lr_p, X_te_p_sc, y_te_p, 'Logistic Regression (Preprocessing)')

### 3.3 Model 2 — Random Forest (Preprocessing)

In [ ]:
rf_p = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_p.fit(X_tr_p_sc, y_tr_p)
res_rf_p = evaluate_model(rf_p, X_te_p_sc, y_te_p, 'Random Forest (Preprocessing)')

### 3.4 Model 3 — Voting Classifier (Preprocessing)

In [ ]:
voting_p = VotingClassifier(estimators=[
    ('lr',  LogisticRegression(random_state=42, max_iter=1000)),
    ('svm', SVC(probability=True, random_state=42)),
    ('knn', KNeighborsClassifier())
], voting='soft')
voting_p.fit(X_tr_p_sc, y_tr_p)
res_voting_p = evaluate_model(voting_p, X_te_p_sc, y_te_p, 'Voting Classifier (Preprocessing)')

### 3.5 Rangkuman Preprocessing Modeling

In [ ]:
results_p = pd.DataFrame([res_lr_p, res_rf_p, res_voting_p]).set_index('model')
print('=== RANGKUMAN PREPROCESSING MODELING ===')
print(results_p.round(4))

results_p.plot(kind='bar', figsize=(11, 5), colormap='Set1', edgecolor='black')
plt.title('Perbandingan Metrik — Preprocessing Modeling', fontsize=13, fontweight='bold')
plt.ylabel('Score'); plt.ylim(0, 1.1)
plt.xticks(rotation=15, ha='right'); plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

---
## 4. Hyperparameter Tuning & Feature Selection
### 4.1 Feature Importance

In [ ]:
rf_fi = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_fi.fit(X_tr_p_sc, y_tr_p)

feat_imp = pd.Series(rf_fi.feature_importances_, index=X_prep.columns)
feat_imp_sorted = feat_imp.sort_values(ascending=False)

print('=== FEATURE IMPORTANCE (TOP 15) ===')
print(feat_imp_sorted.head(15).round(4))

fig, ax = plt.subplots(figsize=(10, 7))
feat_imp_sorted.head(15).plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Feature Importance — Top 15 Fitur', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

TOP_N = 15
top_features = feat_imp_sorted.head(TOP_N).index.tolist()
print(f'\nTop {TOP_N} fitur terpilih: {top_features}')

In [ ]:
# Buat dataset dengan top features
X_tr_sel = pd.DataFrame(X_tr_p_sc, columns=X_prep.columns)[top_features]
X_te_sel = pd.DataFrame(X_te_p_sc,  columns=X_prep.columns)[top_features]
print(f'Shape setelah feature selection: Train={X_tr_sel.shape}, Test={X_te_sel.shape}')

### 4.2 Tuning — Logistic Regression

In [ ]:
rs_lr = RandomizedSearchCV(
    LogisticRegression(random_state=42),
    param_distributions={'C': [0.01, 0.1, 1, 10, 100],
                         'solver': ['lbfgs', 'liblinear'],
                         'max_iter': [500, 1000]},
    n_iter=10, cv=5, scoring='f1', random_state=42, n_jobs=-1, verbose=1
)
rs_lr.fit(X_tr_sel, y_tr_p)
print(f'Best params: {rs_lr.best_params_}')
print(f'Best CV F1 : {rs_lr.best_score_:.4f}')
res_lr_t = evaluate_model(rs_lr.best_estimator_, X_te_sel, y_te_p, 'Logistic Regression (Tuned)')

### 4.3 Tuning — Random Forest

In [ ]:
rs_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions={'n_estimators': [100, 200, 300],
                         'max_depth': [None, 5, 10, 20],
                         'min_samples_split': [2, 5, 10],
                         'min_samples_leaf': [1, 2, 4],
                         'max_features': ['sqrt', 'log2']},
    n_iter=15, cv=5, scoring='f1', random_state=42, n_jobs=-1, verbose=1
)
rs_rf.fit(X_tr_sel, y_tr_p)
print(f'Best params: {rs_rf.best_params_}')
print(f'Best CV F1 : {rs_rf.best_score_:.4f}')
res_rf_t = evaluate_model(rs_rf.best_estimator_, X_te_sel, y_te_p, 'Random Forest (Tuned)')

### 4.4 Tuning — Voting Classifier

In [ ]:
rs_knn = RandomizedSearchCV(
    KNeighborsClassifier(),
    param_distributions={'n_neighbors': [3, 5, 7, 11],
                         'weights': ['uniform', 'distance'],
                         'metric': ['euclidean', 'manhattan']},
    n_iter=8, cv=5, scoring='f1', random_state=42, n_jobs=-1, verbose=1
)
rs_knn.fit(X_tr_sel, y_tr_p)
print(f'Best KNN params: {rs_knn.best_params_}')

best_lr_params = {k: v for k, v in rs_lr.best_params_.items()}
voting_t = VotingClassifier(estimators=[
    ('lr',  LogisticRegression(**best_lr_params, random_state=42)),
    ('svm', SVC(probability=True, random_state=42, C=1)),
    ('knn', rs_knn.best_estimator_)
], voting='soft')
voting_t.fit(X_tr_sel, y_tr_p)
res_voting_t = evaluate_model(voting_t, X_te_sel, y_te_p, 'Voting Classifier (Tuned)')

### 4.5 Rangkuman Semua 9 Model

In [ ]:
all_results = pd.DataFrame([
    res_lr_d,   res_rf_d,   res_voting_d,
    res_lr_p,   res_rf_p,   res_voting_p,
    res_lr_t,   res_rf_t,   res_voting_t
])
all_results['skenario'] = (['Direct']*3 + ['Preprocessing']*3 + ['Tuning']*3)

print('=== RANGKUMAN SEMUA 9 MODEL ===')
print(all_results[['skenario','model','accuracy','precision','recall','f1']].round(4).to_string(index=False))

best_idx  = all_results['f1'].idxmax()
best_info = all_results.iloc[best_idx]
print(f'\n🏆 MODEL TERBAIK : {best_info["model"]}')
print(f'   Accuracy      : {best_info["accuracy"]:.4f}')
print(f'   F1-Score      : {best_info["f1"]:.4f}')

In [ ]:
# Visualisasi F1 semua model
fig, ax = plt.subplots(figsize=(14, 6))
colors_bar = ['#3498db']*3 + ['#2ecc71']*3 + ['#e74c3c']*3
bars = ax.bar(all_results['model'], all_results['f1'], color=colors_bar, edgecolor='black')
ax.set_title('Perbandingan F1-Score — Semua 9 Model', fontsize=14, fontweight='bold')
ax.set_ylabel('F1-Score'); ax.set_ylim(0, 1.1)
plt.xticks(rotation=30, ha='right')
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#3498db', label='Direct'),
    Patch(facecolor='#2ecc71', label='Preprocessing'),
    Patch(facecolor='#e74c3c', label='Tuning')
])
plt.tight_layout(); plt.show()

---
## 5. Simpan Model Terbaik untuk Deployment

In [ ]:
# Tentukan model terbaik berdasarkan F1-score tertinggi
# Map nama model ke objek Python
model_map = {
    'Logistic Regression (Tuned)' : rs_lr.best_estimator_,
    'Random Forest (Tuned)'       : rs_rf.best_estimator_,
    'Voting Classifier (Tuned)'   : voting_t,
    'Logistic Regression (Preprocessing)' : lr_p,
    'Random Forest (Preprocessing)'       : rf_p,
    'Voting Classifier (Preprocessing)'   : voting_p,
    'Logistic Regression (Direct)'        : lr_d,
    'Random Forest (Direct)'              : rf_d,
    'Voting Classifier (Direct)'          : voting_d,
}

best_model_name   = best_info['model']
best_model_object = model_map[best_model_name]

joblib.dump(best_model_object, 'best_model.pkl')
joblib.dump(scaler,            'scaler.pkl')
joblib.dump(top_features,      'features.pkl')

print(f'✅ Model terbaik disimpan: {best_model_name}')
print(f'✅ Scaler disimpan        : scaler.pkl')
print(f'✅ Fitur disimpan         : features.pkl ({len(top_features)} fitur)')
print(f'\nFile siap untuk Streamlit!')

In [ ]:
# (Opsional) Jika di Google Colab — download file pkl
# Hapus tanda # di bawah untuk mengaktifkan
# from google.colab import files
# files.download('best_model.pkl')
# files.download('scaler.pkl')
# files.download('features.pkl')